# ACT Model to OpenVINO IR Conversion

The Action Chunking Transformer (ACT) is a model that learns a generative model over action sequences for bimanual manipulation. See the original paper for details: [Action Chunking Transformer](https://arxiv.org/pdf/2304.13705).

In this tutorial, we show how to convert a Unitree ACT policy (stored in the LeRobot format) to the OpenVINO Intermediate Representation (IR), producing FP32 artifacts.


## Dependency and Core Installation Verification
Run the next cell to verify all thre required packages are installed.

In [ ]:
# Dependency Verification & Core Installation (no auto-install of lerobot)
"""
This cell:
  * Verifies core packages (torch, openvino, nncf + utilities)
  * Installs only missing core packages
  * Checks for lerobot and EXITS with instructions if it's not present

"""
import sys, subprocess, importlib, pathlib, os

CORE_SPECS = [
    'openvino-dev[pot]>=2024.4.0',
    'nncf>=2.14.0',
    'torch>=2.1', 'torchvision', 'accelerate',
    'safetensors', 'numpy', 'pandas', 'matplotlib', 'tqdm', 'h5py',
    'onnx', 'onnxruntime', 'rich'
]
CORE_IMPORTS = {
    'openvino-dev[pot]>=2024.4.0': 'openvino',
    'nncf>=2.14.0': 'nncf',
    'torch>=2.1': 'torch',
    'torchvision': 'torchvision',
    'accelerate': 'accelerate',
    'safetensors': 'safetensors',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'h5py': 'h5py',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'rich': 'rich'
}
SETUP_SCRIPT = pathlib.Path('setup_unitree_lerobot_env.sh')
README_PATH = pathlib.Path('README.md')


def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install'] + list(packages)
    print('[INSTALL]', ' '.join(packages))
    subprocess.check_call(cmd)

print('[CHECK] Core package presence (excluding lerobot)...')
missing = []
for spec, name in CORE_IMPORTS.items():
    try:
        importlib.import_module(name)
        print(f'  [OK] {name}')
    except Exception:
        print(f'  [MISSING] {name} (spec: {spec})')
        missing.append(spec)

if missing:
    print('\n[PHASE] Installing missing core packages...')
    for spec in missing:
        pip_install(spec)
else:
    print('[INFO] All core packages already installed.')

print('\n[RECHECK] Core imports after installation:')
still_missing = []
for spec, name in CORE_IMPORTS.items():
    try:
        importlib.import_module(name)
        print(f'  [OK] {name}')
    except Exception:
        still_missing.append(name)
        print(f'  [FAIL] {name} still missing')
if still_missing:
    print('\n[WARN] Remaining missing core packages:', still_missing)
    print('Try restarting the kernel or checking version conflicts before proceeding.')

print('\n[CHECK] lerobot availability...')
try:
    import lerobot
    print('[OK] lerobot present.')
except Exception as e:
    print('[ERROR] lerobot not importable:', e)
    print('\nACTION REQUIRED:')
    print(f'  Run setup script: bash {SETUP_SCRIPT}')        
    raise SystemExit(1)

print('\n[SUMMARY] Core dependencies verified; lerobot present. Proceed to environment variable export cell.')

[CHECK] Core package presence (excluding lerobot)...
  [OK] openvino
  [OK] nncf
  [OK] torch
  [OK] torchvision
  [OK] accelerate
  [OK] safetensors
  [OK] numpy
  [OK] pandas
  [OK] matplotlib
  [OK] tqdm
  [OK] h5py
  [OK] onnx
  [OK] onnxruntime
  [OK] rich
[INFO] All core packages already installed.

[RECHECK] Core imports after installation:
  [OK] openvino
  [OK] nncf
  [OK] torch
  [OK] torchvision
  [OK] accelerate
  [OK] safetensors
  [OK] numpy
  [OK] pandas
  [OK] matplotlib
  [OK] tqdm
  [OK] h5py
  [OK] onnx
  [OK] onnxruntime
  [OK] rich

[CHECK] lerobot availability...
[OK] lerobot present.

[SUMMARY] Core dependencies verified; lerobot present. Proceed to environment variable export cell.


Next cell configures all the paths.

In [ ]:
# Configuration Parameters (Paths, Precision, Device)
import os, pathlib

CKPT_DIR = pathlib.Path('act_checkpoint') 
NOTEBOOK_DIR = pathlib.Path('.').resolve()
MODEL_DIR = pathlib.Path(os.getenv('ACT_PROJECT_ROOT', NOTEBOOK_DIR))
CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', str(CKPT_DIR / 'model.safetensors')))
IR_OUTPUT_DIR = pathlib.Path(os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs'))
IR_OUTPUT_DIR.mkdir(exist_ok=True)
DATASET_ROOT = pathlib.Path(os.getenv('ACT_DATASET_ROOT', str(MODEL_DIR / 'dataset')))
STATS_PATH = pathlib.Path(os.getenv('ACT_STATS_PATH', str(CKPT_DIR / 'stats.json')))

PRECISIONS = ['FP32', 'FP16']
TARGET_DEVICE = os.getenv('ACT_TARGET_DEVICE', 'CPU')

print('Notebook directory:', NOTEBOOK_DIR)
print('Relative checkpoint dir:', CKPT_DIR)
print('Resolved checkpoint file path:', CHECKPOINT_PATH)
print('Dataset root:', DATASET_ROOT)
print('Stats path (may not exist yet):', STATS_PATH)
print('Output directory:', IR_OUTPUT_DIR)
print('Target device:', TARGET_DEVICE)

## Acquire ACT Checkpoint Assets
Download the ACT model artifacts: `model.safetensors`, `config.json`, and `train_config.json` into `act_checkpoint/`

In [ ]:
# Export environment variables
import os, pathlib
CKPT_DIR = pathlib.Path('act_checkpoint')
os.environ['ACT_CHECKPOINT'] = str(CKPT_DIR / 'model.safetensors')
os.environ['ACT_CONFIG_PATH'] = str(CKPT_DIR / 'config.json')
os.environ['ACT_TRAIN_CONFIG_PATH'] = str(CKPT_DIR / 'train_config.json')
stats_path = CKPT_DIR / 'stats.json'
if stats_path.exists():
    os.environ['ACT_STATS_PATH'] = str(stats_path)
print('[INFO] Checkpoint directory (relative):', CKPT_DIR)
print('[INFO] Environment variables:')
for k in ['ACT_CHECKPOINT','ACT_CONFIG_PATH','ACT_TRAIN_CONFIG_PATH','ACT_STATS_PATH']:
    if k in os.environ:
        print('  ', k, '=', os.environ[k])


[INFO] Checkpoint directory (relative): act_checkpoint
[INFO] Environment variables:
   ACT_CHECKPOINT = act_checkpoint/model.safetensors
   ACT_CONFIG_PATH = act_checkpoint/config.json
   ACT_TRAIN_CONFIG_PATH = act_checkpoint/train_config.json
   ACT_STATS_PATH = act_checkpoint/stats.json


In [ ]:
# Load Original ACT Model
import os, json, inspect, pathlib, sys, importlib
from safetensors.torch import load_file
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

CHECKPOINT_PATH = pathlib.Path(os.getenv('ACT_CHECKPOINT', 'act_checkpoint/model.safetensors'))
CONFIG_PATH = pathlib.Path(os.getenv('ACT_CONFIG_PATH', str(CHECKPOINT_PATH.parent / 'config.json')))

print('[LOAD] CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('[LOAD] CONFIG_PATH     =', CONFIG_PATH)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint file not found at {CHECKPOINT_PATH}.\n"
        "Ensure you have: (1) placed model.safetensors in act_checkpoint/, or (2) set ACT_CHECKPOINT env var, then re-run this cell."
    )
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"config.json not found at {CONFIG_PATH}.\n"
        "Place config.json next to the checkpoint (act_checkpoint/config.json) or set ACT_CONFIG_PATH."
    )

with open(CONFIG_PATH, 'r') as f:
    cfg_dict = json.load(f)

# Filter config keys to ACTConfig signature
valid_keys = set(inspect.signature(ACTConfig.__init__).parameters.keys()); valid_keys.discard('self')
filtered_cfg = {k: v for k, v in cfg_dict.items() if k in valid_keys}

# Helper wrappers
def wrap_features(feat_dict):
    return {k: PolicyFeature(type=FeatureType(v['type']), shape=tuple(v['shape'])) for k, v in feat_dict.items()}

def wrap_norm_map(norm_map):
    return {FeatureType(k): NormalizationMode(v) for k, v in norm_map.items()}

if 'input_features' in filtered_cfg:
    filtered_cfg['input_features'] = wrap_features(filtered_cfg['input_features'])
if 'output_features' in filtered_cfg:
    filtered_cfg['output_features'] = wrap_features(filtered_cfg['output_features'])
if 'normalization_mapping' in filtered_cfg:
    filtered_cfg['normalization_mapping'] = wrap_norm_map(filtered_cfg['normalization_mapping'])

act_config = ACTConfig(**filtered_cfg)
act_config.use_vae = False
policy = ACTPolicy(act_config)
weights = load_file(str(CHECKPOINT_PATH))
policy.load_state_dict(weights, strict=False)
policy.eval()
print('Loaded ACTPolicy from safetensors. Params:', sum(p.numel() for p in policy.parameters()))

# Extract dimensions
action_dim = filtered_cfg['output_features']['action'].shape[0]
chunk_size = filtered_cfg.get('chunk_size', 100)
# Camera keys
camera_keys = sorted([k for k in cfg_dict['input_features'] if k.startswith('observation.images.')])
print('Detected cameras:', camera_keys)


[LOAD] CHECKPOINT_PATH = act_checkpoint/model.safetensors
[LOAD] CONFIG_PATH     = act_checkpoint/config.json
Loaded ACTPolicy from safetensors. Params: 34246684
Detected cameras: ['observation.images.cam_left_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_high', 'observation.images.cam_right_wrist']


In [ ]:
# Inspect Model Architecture and Construct Full Dummy Inputs
import torch 

state_dim = policy.config.input_features['observation.state'].shape[0]
chunk_size = chunk_size  # from previous cell
H, W = 480, 640
cams = camera_keys

# Use shapes from config if specified
image_tensors = []
for cam in cams:
    shape = policy.config.input_features[cam].shape  # e.g. [3, H, W]
    img = torch.zeros(1, *shape, dtype=torch.float32)
    image_tensors.append(img)

state = torch.zeros(1, state_dim, dtype=torch.float32)
action_is_pad = torch.zeros(1, chunk_size, dtype=torch.bool)
action_seq = torch.zeros(1, chunk_size, action_dim, dtype=torch.float32)

env_state = None
if 'observation.environment_state' in policy.config.input_features:
    env_dim = policy.config.input_features['observation.environment_state'].shape[0]
    env_state = torch.zeros(1, env_dim, dtype=torch.float32)

print('State shape:', state.shape)
print('Image shapes:', [t.shape for t in image_tensors])
print('Action pad shape:', action_is_pad.shape)
print('Action seq shape:', action_seq.shape)
if env_state is not None:
    print('Environment state shape:', env_state.shape)


State shape: torch.Size([1, 28])
Image shapes: [torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640])]
Action pad shape: torch.Size([1, 100])
Action seq shape: torch.Size([1, 100, 28])


In [ ]:
# Prepare Ordered Inputs
# Order: observation.state, each camera image, action_is_pad, action, optional environment_state
ordered_inputs = [state] + image_tensors + [action_is_pad, action_seq] + ([env_state] if env_state is not None else [])
print('Ordered input tensor shapes:', [t.shape for t in ordered_inputs])


Ordered input tensor shapes: [torch.Size([1, 28]), torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640]), torch.Size([1, 3, 480, 640]), torch.Size([1, 100]), torch.Size([1, 100, 28])]


In [ ]:
# Export ACT Model to ONNX
import torch, inspect, os, pathlib

# Ensure IR_OUTPUT_DIR is available
try:
    IR_OUTPUT_DIR
except NameError:
    IR_OUTPUT_DIR = pathlib.Path(os.getenv('ACT_IR_OUTPUT_DIR', 'openvino_ir_outputs'))
    print('[ONNX] IR_OUTPUT_DIR was undefined; set to', IR_OUTPUT_DIR)

# Create directory if missing
IR_OUTPUT_DIR.mkdir(exist_ok=True)

# Build ONNXWrapper
class ONNXWrapper(torch.nn.Module):
    def __init__(self, model, camera_keys):
        super().__init__()
        self.model = model
        self.camera_keys = camera_keys
    def forward(self, observation_state, *cam_inputs_and_rest):
        num_cams = len(self.camera_keys)
        cam_inputs = cam_inputs_and_rest[:num_cams]
        action_is_pad_local = cam_inputs_and_rest[num_cams]
        action_local = cam_inputs_and_rest[num_cams + 1]
        observation_environment_state = None
        if len(cam_inputs_and_rest) > num_cams + 2:
            observation_environment_state = cam_inputs_and_rest[num_cams + 2]
        batch = {'observation.state': observation_state}
        for i, cam_key in enumerate(self.camera_keys):
            batch[cam_key] = cam_inputs[i]
        batch['action_is_pad'] = action_is_pad_local
        batch['action'] = action_local
        batch['observation.images'] = list(cam_inputs)
        if observation_environment_state is not None:
            batch['observation.environment_state'] = observation_environment_state
        prediction = self.model.model(batch)
        if isinstance(prediction, tuple):
            prediction = prediction[0]
        return prediction

onnx_path = IR_OUTPUT_DIR / 'model.onnx'
if not onnx_path.exists():
    # Construct dummy args & input names
    dummy_args = [torch.randn_like(state)]
    input_names = ['observation_state']
    for i, cam_key in enumerate(camera_keys):
        cam_tensor = torch.randn_like(image_tensors[i])
        dummy_args.append(cam_tensor)
        input_names.append(f'observation_images_{i}')
    dummy_args.append(torch.zeros_like(action_is_pad))
    dummy_args.append(torch.zeros_like(action_seq))
    input_names += ['action_is_pad', 'action']
    if env_state is not None:
        dummy_args.append(torch.randn_like(env_state))
        input_names.append('observation_environment_state')

    torch.onnx.export(
        ONNXWrapper(policy, camera_keys),
        tuple(dummy_args),
        str(onnx_path),
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=input_names,
        output_names=['output']
    )
    print('ONNX model exported to', onnx_path)
else:
    print('ONNX already exists:', onnx_path)


/home/case/Mohammad/notebook/openvino_notebooks/notebooks/lerobot_act/unitree_IL_lerobot/unitree_lerobot/lerobot/src/lerobot/policies/act/modeling_act.py:478: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  encoder_in_pos_embed = list(self.encoder_1d_feature_pos_embed.weight.unsqueeze(1))
/home/case/Mohammad/notebook/openvino_notebooks/notebooks/lerobot_act/unitree_IL_lerobot/unitree_lerobot/lerobot/src/lerobot/policies/act/modeling_act.py:478: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  encoder_in_pos_embed = list(self.encoder_1d_feature_pos_embed.weight.unsqueeze(1))
/home/case/Mohammad/notebook/openvino_notebooks/notebooks/lerobot_a

ONNX model exported to openvino_ir_outputs/model.onnx


In [ ]:
# Convert ONNX to OpenVINO IR
import subprocess, shlex, os, pathlib, shutil
MO_OUT_DIR = IR_OUTPUT_DIR
IR_FP32_XML = MO_OUT_DIR / 'act_model_fp32.xml'
IR_FP32_BIN = MO_OUT_DIR / 'act_model_fp32.bin'

if IR_FP32_XML.exists() and IR_FP32_BIN.exists():
    print('IR already present, skipping MO conversion:', IR_FP32_XML)
else:
    cmd = f"mo --input_model {IR_OUTPUT_DIR / 'model.onnx'} --output_dir {MO_OUT_DIR} --compress_to_fp16=False"
    print('Running Model Optimizer:', cmd)
    try:
        subprocess.run(shlex.split(cmd), check=True)
        src_xml = MO_OUT_DIR / 'model.xml'
        src_bin = MO_OUT_DIR / 'model.bin'
        if src_xml.exists():
            shutil.copy(src_xml, IR_FP32_XML)
        if src_bin.exists():
            shutil.copy(src_bin, IR_FP32_BIN)
        print('MO conversion complete. Standardized IR files:', IR_FP32_XML, IR_FP32_BIN)
    except Exception as e:
        raise RuntimeError(f'Model Optimizer failed. Ensure openvino-dev installed. Original error: {e}')

Running Model Optimizer: mo --input_model openvino_ir_outputs/model.onnx --output_dir openvino_ir_outputs --compress_to_fp16=False
[ INFO ] MO command line tool is considered as the legacy conversion API as of OpenVINO 2023.2 release.
In 2025.0 MO command line tool and openvino.tools.mo.convert_model() will be removed. Please use OpenVINO Model Converter (OVC) or openvino.convert_model(). OVC represents a lightweight alternative of MO and provides simplified model conversion API. 
Find more information about transition from MO to OVC at https://docs.openvino.ai/2023.2/openvino_docs_OV_Converter_UG_prepare_model_convert_model_MO_OVC_transition.html
Check for a new version of Intel(R) Distribution of OpenVINO(TM) toolkit here https://software.intel.com/content/www/us/en/develop/tools/openvino-toolkit/download.html?cid=other&source=prod&campid=ww_2023_bu_IOTG_OpenVINO-2023-1&content=upg_all&medium=organic or on https://github.com/openvinotoolkit/openvino
[ SUCCESS ] Generated IR version 1

In [ ]:
# Confirm IR Files Produced by MO
print('The following IR files present:')
for f in [IR_OUTPUT_DIR / 'act_model_fp32.xml', IR_OUTPUT_DIR / 'act_model_fp32.bin']:
    print('-', f.name, 'exists' if f.exists() else 'MISSING', '| size:', f.stat().st_size if f.exists() else 0)


The following IR files present:
- act_model_fp32.xml MISSING | size: 0
- act_model_fp32.bin MISSING | size: 0
